# Chapter 3.1 · Retrieval and RAG

The firm's full compliance handbook, chunked, embedded and searched four ways, then an agent that searches it for itself. The job is the one from the chapter: find the clause that answers an employee's question, and cite it.

**Before you run:** the same three steps as chapter 1. Sign in at [lumen.ncsa.illinois.edu](https://lumen.ncsa.illinois.edu/chat), create an API key on your [profile page](https://lumen.ncsa.illinois.edu/profile), and add it to Colab as a secret named `LUMEN_API_KEY` with notebook access enabled.

The model is `glm-5.3-flash` on Lumen, the campus LLM service.

In [ ]:
%pip install -q openai rank-bm25 sentence-transformers

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

MODEL = "glm-5.3-flash"  # highly preferred for this workshop
client = OpenAI(base_url="https://lumen.ncsa.illinois.edu/v1", api_key=userdata.get("LUMEN_API_KEY"))

## The handbook

The same handbook the textbook uses: 73 clauses in 15 sections, each clause with an id you can cite. It is downloaded from the book's repository.

In [ ]:
import urllib.request, importlib
urllib.request.urlretrieve("https://raw.githubusercontent.com/ashcastelinocs124/msba-ai-workshop/main/book/_static/py/handbook.py", "handbook.py")
import handbook; importlib.reload(handbook)
from handbook import CLAUSES, HANDBOOK

print(len(CLAUSES), "clauses,", len(HANDBOOK.split()), "words")
print(CLAUSES[5]["id"], "·", CLAUSES[5]["text"])

## Chunking

Fixed-size chunks, with and without overlap, against the handbook's own structure (one clause per chunk). Watch what happens to the blackout rule.

In [ ]:
def chunk(text, size, overlap=0):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("size must be positive and overlap smaller than size")
    words, step = text.split(), size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

rule = "within 14 days before or after the firm publishes research on it"
for size, overlap in [(15, 0), (15, 5), (40, 10), (400, 0)]:
    pieces = chunk(HANDBOOK, size, overlap)
    home = [p for p in pieces if rule in p]
    status = f"whole, inside a {len(home[0].split())}-word chunk" if home else "cut in half"
    print(f"{size} words, overlap {overlap}: {len(pieces)} chunks. The blackout rule is {status}.")

## Embeddings

The book embedded the handbook with Azure's `text-embedding-3-small`. Here the notebook asks Lumen whether it offers an embedding model; if not, it uses `all-MiniLM-L6-v2`, a small open model that runs on Colab's own processor. The numbers differ between models; the idea does not.

In [ ]:
import numpy as np

lumen_embedders = [m.id for m in client.models.list().data if "embed" in m.id.lower()]
if lumen_embedders:
    EMBEDDER = lumen_embedders[0]
    def embed(texts):
        out = client.embeddings.create(model=EMBEDDER, input=texts)
        v = np.array([d.embedding for d in out.data])
        return v / np.linalg.norm(v, axis=1, keepdims=True)
else:
    from sentence_transformers import SentenceTransformer
    EMBEDDER = "all-MiniLM-L6-v2 (on Colab)"
    _st = SentenceTransformer("all-MiniLM-L6-v2")
    def embed(texts):
        return np.array(_st.encode(texts, normalize_embeddings=True))

print("Embedding with:", EMBEDDER)
CHUNK_VECS = embed([c["text"] for c in CLAUSES])
print(CHUNK_VECS.shape[0], "clauses, each turned into", CHUNK_VECS.shape[1], "numbers")

In [ ]:
q = "Can I trade during the quiet period after we publish?"
v = embed([q])[0]
for cid in ["personal-trading-2", "gifts-4", "expense-2"]:
    i = next(i for i, c in enumerate(CLAUSES) if c["id"] == cid)
    print(f"similarity to {cid}: {float(v @ CHUNK_VECS[i]):.3f}")

## Four ways to retrieve

Keyword (BM25), semantic (cosine similarity of embeddings), hybrid (reciprocal rank fusion of the two), and a reranker that asks the model to score each of hybrid's top ten.

In [ ]:
import re
from rank_bm25 import BM25Okapi

STOP = {"the", "a", "an", "of", "to", "in", "on", "for", "and", "or", "is", "are", "be", "can", "i", "my", "we", "our",
        "it", "any", "after", "before", "what", "does", "do", "may", "must", "with", "at", "by", "from", "how", "if"}
def terms(text):
    words = (w.strip(".") for w in re.findall(r"[a-z0-9$@.]+", text.lower()))   # "sold." at the end of a clause is "sold"
    return [w for w in words if w and w not in STOP]

_bm25 = BM25Okapi([terms(c["text"]) for c in CLAUSES])

def _ranked(scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return [{"id": CLAUSES[i]["id"], "text": CLAUSES[i]["text"], "score": round(float(scores[i]), 3)} for i in order]

def keyword(q, k=3):
    return _ranked(_bm25.get_scores(terms(q)), k)

def semantic(q, k=3):
    return _ranked(CHUNK_VECS @ embed([q])[0], k)

def hybrid(q, k=3, depth=10, c=60):
    fused = {}
    for ranking in (keyword(q, depth), semantic(q, depth)):
        for rank, r in enumerate(ranking):
            fused[r["id"]] = fused.get(r["id"], 0) + 1 / (c + rank + 1)
    ids = sorted(fused, key=lambda i: -fused[i])[:k]
    by_id = {c["id"]: c for c in CLAUSES}
    return [{"id": i, "text": by_id[i]["text"], "score": round(fused[i], 4)} for i in ids]

def show(results):
    for i, r in enumerate(results, 1):
        print(f"{i}. [{r['id']}] ({r['score']}) {r['text'][:90]}")

In [ ]:
import json

def rerank(q, k=3):
    """Ask the model to score each of hybrid's top ten for how well it answers the question, 0 to 10."""
    cands = hybrid(q, 10)
    listing = "\n".join(f"{c['id']}: {c['text']}" for c in cands)
    reply = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "You rank handbook clauses. Reply with JSON only: an object mapping each clause id to a score from 0 (irrelevant) to 10 (answers the question directly)."},
        {"role": "user", "content": f"Question: {q}\n\nClauses:\n{listing}"}])
    text = reply.choices[0].message.content or ""
    try:
        scores = json.loads(text[text.index("{"): text.rindex("}") + 1])
    except ValueError:
        print("(The reranker did not return JSON; keeping hybrid's order.)")
        return cands[:k]
    for c in cands:
        c["score"] = float(scores.get(c["id"], 0))
    return sorted(cands, key=lambda c: -c["score"])[:k]

METHODS = {"keyword": keyword, "semantic": semantic, "hybrid": hybrid, "rerank": rerank}

In [ ]:
QUESTIONS = [
    ("Can I trade during the quiet period after we publish?", "personal-trading-2"),
    ("Can I short NVIDIA?", "personal-trading-6"),
    ("Is a $150 bottle of wine from a vendor OK?", "gifts-1"),
    ("Who approves my trade?", "pre-clearance-4"),
    ("Do I need permission to buy an S&P 500 index fund?", "personal-trading-7"),
]
for q, want in QUESTIONS:
    firsts = {m: f(q, 1)[0]["id"] for m, f in METHODS.items()}
    marks = "  ".join(f"{m}: {'✓' if firsts[m] == want else '✗ ' + firsts[m]}" for m in METHODS)
    print(f"{q}\n   right clause {want}.  {marks}\n")

## An agent that searches for itself

The chapter 1 loop with one tool, `search_handbook`. The agent decides what to search for, and whether to search again. It prints one `Step N:` line per step, like the textbook.

In [ ]:
TOOLS = [{"type": "function", "function": {
    "name": "search_handbook",
    "description": "Search the firm's compliance handbook. Returns clauses with ids to cite. If none answers the question, rewrite the query in the handbook's own words and search again.",
    "parameters": {"type": "object", "properties": {
        "query": {"type": "string"},
        "method": {"type": "string", "enum": ["keyword", "semantic", "hybrid"]}},
        "required": ["query"]}}}]

SYSTEM = ("You answer Champaign Capital employees' questions about the compliance handbook. Search before answering. "
          "Answer only from clauses you found, and end with the id you relied on as [source: id]. "
          "You recommend; only a compliance officer approves a trade.")

def agent(question, max_steps=5):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        msg = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS).choices[0].message
        if not msg.tool_calls:
            print(f"Step {step}: answered — {msg.content}")
            return msg.content
        call = msg.tool_calls[0]
        args = json.loads(call.function.arguments or "{}")
        method = args.get("method") if args.get("method") in METHODS else "hybrid"   # a model can ask for a method that does not exist
        hits = METHODS[method](args["query"], 3)
        print(f"Step {step}: searched handbook ({args['query']}, {method}) → {', '.join(h['id'] for h in hits)}")
        msgs.append({"role": "assistant", "content": None, "tool_calls": [{"id": call.id, "type": "function",
                     "function": {"name": call.function.name, "arguments": call.function.arguments}}]})
        msgs.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps([{"id": h["id"], "text": h["text"]} for h in hits])})
    print("Stopped: step budget exhausted.")

answer = agent("Can I dump my Caterpillar shares a week after buying them?")

In [ ]:
answer = agent("Can Tom sell his Deere shares 6 days after we published research on Deere?")

## Exercise

1. Find a question of your own where `hybrid` ranks the right clause first and `semantic` does not. Then find one where `rerank` changes the top answer. Write down why, in a sentence each.
2. Re-chunk the handbook with fixed 40-word chunks and a 10-word overlap (the `chunk` function above), embed those chunks instead of the clauses, and run the five `QUESTIONS` again. Which ones got worse, and why?
3. After `agent` answers, add a check in code that the cited id exists in `CLAUSES` and that the clause shares at least one key term with the answer. What does the check catch, and what can it not catch?